In [0]:
from pyspark.sql import Row, functions as F, types as T
from delta.tables import DeltaTable
from datetime import datetime
from zoneinfo import ZoneInfo
import requests
import json
import time
import logging
import uuid

In [0]:
# ---------------------------------------------------------------------------
# Logging Configuration
# ---------------------------------------------------------------------------
NOTEBOOK_NAME    = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split("/")[-1]

log              = logging.getLogger(NOTEBOOK_NAME)
log.setLevel(logging.INFO)
if not log.handlers:
    _handler     = logging.StreamHandler()
    _handler.setFormatter(
        logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
    )
    log.addHandler(_handler)

# ---------------------------------------------------------------------------
# Dynamic Configuration
# ---------------------------------------------------------------------------
dbutils.widgets.dropdown("env", "dev", ["dev", "tst", "prd"], "Environment")
env         = dbutils.widgets.get("env")
CATALOG          = f"{env}_corpdatav2"
BRONZE_DB        = f"{CATALOG}.bronze"
SILVER_DB        = f"{CATALOG}.silver"
AUDIT_CATALOG    = f"{env}_audit"
LOGS             = f"{AUDIT_CATALOG}.logs"

LOGS_TABLE       = f"{LOGS}.{NOTEBOOK_NAME}"
TARGET_BRONZE    = f"{BRONZE_DB}.SF_API_UserAccount"
TARGET_SILVER    = f"{SILVER_DB}.SF_API_UserAccount"

BASE_APIM        = "https://apim.api-dev.seaspancorp.com"
SESSION_APIM     = f"{BASE_APIM}/sapsf-prod/oauth/idp"
AUTH_APIM        = f"{BASE_APIM}/sapsf-prod/oauth/token"
EMPLOYEE_APIM    = f"""{BASE_APIM}/sapsf-prod/odata/v2/PerPersonal?fromDate=1990-01-01&$select=personIdExternal,personNav%2FemploymentNav%2FstartDate,personNav%2FemploymentNav%2FuserId,firstName,lastName,lastModifiedDateTime,personNav%2FemploymentNav%2FendDate,createdDateTime,personNav%2FemploymentNav%2FisContingentWorker,personNav%2FemploymentNav%2FlastModifiedDateTime,personNav%2FemploymentNav%2FemploymentId,personNav%2FemploymentNav%2FjobInfoNav%2FemplStatusNav%2FexternalCode,personNav%2FemploymentNav%2FjobInfoNav%2FbusinessUnit,personNav%2FemploymentNav%2FjobInfoNav%2FlastModifiedDateTime,personNav%2FemploymentNav%2FjobInfoNav%2FjobTitle,personNav%2FemploymentNav%2FjobInfoNav%2FpositionNav%2FexternalName_en_US,personNav%2FemploymentNav%2FjobInfoNav%2Flocation,personNav%2FemploymentNav%2FjobInfoNav%2Fdepartment,personNav%2FemploymentNav%2FjobInfoNav%2FlocationNav%2FlastModifiedDateTime,personNav%2FemploymentNav%2FjobInfoNav%2FlocationNav%2Fname,personNav%2FemploymentNav%2FjobInfoNav%2FmanagerUserNav%2Femail,personNav%2FemploymentNav%2FjobInfoNav%2FjobCodeNav%2FlastModifiedDateTime,personNav%2FemploymentNav%2FjobInfoNav%2FjobCodeNav%2Fname,personNav%2FemploymentNav%2FjobInfoNav%2FdepartmentNav%2FlastModifiedDateTime,personNav%2FemploymentNav%2FjobInfoNav%2FdepartmentNav%2Fname,personNav%2FemploymentNav%2FjobInfoNav%2Fdepartment%2Fname,personNav%2FemploymentNav%2FjobInfoNav%2FcompanyNav%2FlastModifiedDateTime,personNav%2FemploymentNav%2FjobInfoNav%2FcompanyNav%2Fname,personNav%2FuserAccountNav%2FtimeZone,personNav%2FemailNav%2FemailAddress,personNav%2FemailNav%2FemailType,personNav%2FpersonEmpTerminationInfoNav%2FlatestTerminationDate,personNav%2FemploymentNav%2FjobInfoNav%2FcostCenter,personNav%2FemploymentNav%2FjobInfoNav%2Fdivision,personNav%2FemploymentNav%2FjobInfoNav%2FemployeeClass,personNav%2FemploymentNav%2FemploymentId&$filter=%28personNav%2FemploymentNav%2FjobInfoNav%2Fcompany+eq+%2700500%27+or+personNav%2FemploymentNav%2FjobInfoNav%2Fcompany+eq+%2700300%27+or+personNav%2FemploymentNav%2FjobInfoNav%2Fcompany+eq+%2701100%27+or+personNav%2FemploymentNav%2FjobInfoNav%2Fcompany+eq+%2701101%27+or+personNav%2FemploymentNav%2FjobInfoNav%2Fcompany+eq+%2700100%27+or+personNav%2FemploymentNav%2FjobInfoNav%2Fcompany+eq+%2700200%27+or+personNav%2FemploymentNav%2FjobInfoNav%2Fcompany+eq+%2700504%27+or+personNav%2FemploymentNav%2FjobInfoNav%2Fcompany+eq+%2700502%27+or+personNav%2FemploymentNav%2FjobInfoNav%2Fcompany+eq+%2700501%27+or+personNav%2FemploymentNav%2FjobInfoNav%2Fcompany+eq+%2700503%27%29&toDate=9999-12-31&$expand=personNav%2FemploymentNav%2FjobInfoNav,personNav%2FemploymentNav%2FjobInfoNav%2FcompanyNav,personNav%2FemploymentNav,personNav,personNav%2FemploymentNav%2FjobInfoNav%2FdepartmentNav,personNav%2FemploymentNav%2FjobInfoNav%2FjobCodeNav,personNav%2FemploymentNav%2FjobInfoNav%2FlocationNav,personNav%2FemploymentNav%2FjobInfoNav%2FmanagerUserNav,personNav%2FemailNav,personNav%2FemploymentNav%2FjobInfoNav%2FemplStatusNav,personNav%2FuserAccountNav,personNav%2FemploymentNav%2FjobInfoNav%2FpositionNav,personNav%2FpersonEmpTerminationInfoNav&$format=JSON&$skiptoken="""

APIKEY           = dbutils.secrets.get(scope="azure-scope", key="adb-sf-apim")

HEADERS          = {
    "Content-Type": "application/x-www-form-urlencoded",
    "apiKey": APIKEY
}


# ---------------------------------------------------------------------------
# Pipeline Settings
# ---------------------------------------------------------------------------
MAX_RETRIES      = 3
BATCH_RUN_ID     = str(uuid.uuid4())

log.info(f"Environment: {env}")
log.info(f"Batch Run ID: {BATCH_RUN_ID}")
log.info(f"Catalog: {CATALOG}")
log.info(f"Max Retries: {MAX_RETRIES}")

In [0]:
audit_schema = T.StructType([
    T.StructField("EventID", T.StringType(), True),
    T.StructField("batch_run_id", T.StringType(), True),
    T.StructField("api_type", T.StringType(), True),
    T.StructField("http_method", T.StringType(), True),
    T.StructField("api_url", T.StringType(), True),
    T.StructField("input_value", T.StringType(), True),
    T.StructField("request_body", T.StringType(), True),
    T.StructField("attempt_number", T.IntegerType(), True),
    T.StructField("request_timestamp", T.TimestampType(), True),
    T.StructField("response_code", T.IntegerType(), True),
    T.StructField("response_body", T.StringType(), True),
    T.StructField("duration_ms", T.LongType(), True),
    T.StructField("status", T.StringType(), True),
    T.StructField("error_message", T.StringType(), True),
    T.StructField("execution_context", T.StringType(), True),
])

bronze_schema = T.StructType([
    T.StructField("Raw_data", T.StringType(), True),
    T.StructField("Api_metadata", T.StringType(), True),
    T.StructField("_is_normalized", T.BooleanType(), True),
    T.StructField("_ingestion_ts", T.TimestampType(), True),
])

silver_schema = T.StructType([
    T.StructField("UserId", T.IntegerType(), True),
    T.StructField("firstName", T.StringType(), True),
    T.StructField("lastName", T.StringType(), True),
    T.StructField("busEmail", T.StringType(), True),
    T.StructField("empStatus", T.StringType(), True),
    T.StructField("isContingentWorker", T.StringType(), True),
    T.StructField("createdDateTime", T.TimestampType(), True),
    T.StructField("lastModifiedDateTime", T.TimestampType(), True),
    T.StructField("startDate", T.StringType(), True),
    T.StructField("timeZone", T.StringType(), True),
    T.StructField("businessUnit", T.StringType(), True),
    T.StructField("jobInfoNavLastModifiedDateTime", T.TimestampType(), True),
    T.StructField("endDate", T.StringType(), True),
    T.StructField("managerEmail", T.StringType(), True),
    T.StructField("jobTitle", T.StringType(), True),
    T.StructField("positionTitle", T.StringType(), True),
    T.StructField("location", T.StringType(), True),
    T.StructField("department", T.StringType(), True),
    T.StructField("locationNavLastModifiedDateTime", T.TimestampType(), True),
    T.StructField("locationNavName", T.StringType(), True),
    T.StructField("jobCodeNavLastModifiedDateTime", T.TimestampType(), True),
    T.StructField("jobCodeName", T.StringType(), True),
    T.StructField("departmentNavLastModifiedDateTime", T.TimestampType(), True),
    T.StructField("departmentNavName", T.StringType(), True),
    T.StructField("companyNavLastModifiedDateTime", T.TimestampType(), True),
    T.StructField("companyNavName", T.StringType(), True),
    T.StructField("employmentNavLastModifiedDateTime", T.TimestampType(), True),
    T.StructField("terminationDate", T.StringType(), True),
    T.StructField("costCenter", T.StringType(), True),
    T.StructField("division", T.StringType(), True),
    T.StructField("employeeClass", T.StringType(), True),
    T.StructField("employeeId", T.StringType(), True),
    T.StructField("LoadDateUTC", T.TimestampType(), True),
    T.StructField("PersonIdExternal", T.IntegerType(), True),
])

# OData record struct (single employee record from d.results)
_odata_record_struct = T.StructType([
    T.StructField("personIdExternal", T.StringType()),
    T.StructField("firstName", T.StringType()),
    T.StructField("lastName", T.StringType()),
    T.StructField("createdDateTime", T.StringType()),
    T.StructField("lastModifiedDateTime", T.StringType()),
    T.StructField("personNav", T.StructType([
        T.StructField("employmentNav", T.StructType([
            T.StructField("results", T.ArrayType(T.StructType([
                T.StructField("userId", T.StringType()),
                T.StructField("startDate", T.StringType()),
                T.StructField("endDate", T.StringType()),
                T.StructField("isContingentWorker", T.StringType()),
                T.StructField("lastModifiedDateTime", T.StringType()),
                T.StructField("employmentId", T.StringType()),
                T.StructField("jobInfoNav", T.StructType([
                    T.StructField("results", T.ArrayType(T.StructType([
                        T.StructField("businessUnit", T.StringType()),
                        T.StructField("lastModifiedDateTime", T.StringType()),
                        T.StructField("jobTitle", T.StringType()),
                        T.StructField("location", T.StringType()),
                        T.StructField("department", T.StringType()),
                        T.StructField("costCenter", T.StringType()),
                        T.StructField("division", T.StringType()),
                        T.StructField("employeeClass", T.StringType()),
                        T.StructField("emplStatusNav", T.StructType([
                            T.StructField("externalCode", T.StringType()),
                        ])),
                        T.StructField("positionNav", T.StructType([
                            T.StructField("externalName_en_US", T.StringType()),
                        ])),
                        T.StructField("managerUserNav", T.StructType([
                            T.StructField("email", T.StringType()),
                        ])),
                        T.StructField("locationNav", T.StructType([
                            T.StructField("lastModifiedDateTime", T.StringType()),
                            T.StructField("name", T.StringType()),
                        ])),
                        T.StructField("jobCodeNav", T.StructType([
                            T.StructField("lastModifiedDateTime", T.StringType()),
                            T.StructField("name", T.StringType()),
                        ])),
                        T.StructField("departmentNav", T.StructType([
                            T.StructField("lastModifiedDateTime", T.StringType()),
                            T.StructField("name", T.StringType()),
                        ])),
                        T.StructField("companyNav", T.StructType([
                            T.StructField("lastModifiedDateTime", T.StringType()),
                            T.StructField("name", T.StringType()),
                        ])),
                    ])))
                ])),
            ])))
        ])),
        T.StructField("userAccountNav", T.StructType([
            T.StructField("timeZone", T.StringType()),
        ])),
        T.StructField("emailNav", T.StructType([
            T.StructField("results", T.ArrayType(T.StructType([
                T.StructField("emailAddress", T.StringType()),
                T.StructField("emailType", T.StringType()),
            ])))
        ])),
        T.StructField("personEmpTerminationInfoNav", T.StructType([
            T.StructField("latestTerminationDate", T.StringType()),
        ])),
    ])),
])

# Page-level schema: Raw_data is a single API page response per bronze row
# Structure: {d: {results: [record, ...], __next: "url"} }
_odata_page_schema = T.StructType([
    T.StructField("d", T.StructType([
        T.StructField("results", T.ArrayType(_odata_record_struct)),
        T.StructField("__next", T.StringType()),
    ]))
])

In [0]:
# ---------------------------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------------------------
def call_api(url, method="POST", headers=None, body=None, data=None, params=None, api_type="sapsf", input_value=None):
    """Pure API caller with retry logic for single-hit endpoints.
    Use `body` for JSON payloads, `data` for form-encoded payloads.
    Returns result dict with all attempts attached.
    Raises RuntimeError on HTTP 400-428 after max retries.
    """
    attempts = []
    last_status_code = None

    for attempt in range(1, MAX_RETRIES + 1):
        event_id = str(uuid.uuid4())
        start_ts = time.time()
        request_ts = datetime.now()

        try:
            response = requests.request(
                method, url, headers=headers, json=body, data=data, params=params, timeout=60
            )
            duration_ms = int((time.time() - start_ts) * 1000)
            last_status_code = response.status_code

            audit_body = json.dumps(body or data) if (body or data) else None

            if 200 <= response.status_code <= 204:
                # Safely parse response — return raw text if not valid JSON
                try:
                    response_data = response.json() if response.text else None
                except (ValueError, json.JSONDecodeError):
                    response_data = response.text

                attempts.append({
                    "EventID": event_id,
                    "batch_run_id": BATCH_RUN_ID,
                    "api_type": api_type,
                    "http_method": method,
                    "api_url": url,
                    "input_value": input_value,
                    "request_body": audit_body,
                    "attempt_number": attempt,
                    "request_timestamp": request_ts,
                    "response_code": response.status_code,
                    "response_body": response.text[:4000],
                    "duration_ms": duration_ms,
                    "status": "SUCCESS",
                    "error_message": None,
                    "execution_context": None,
                })
                return {
                    "success": True,
                    "response_data": response_data,
                    "api_metadata": {
                        "Status": "SUCCESS",
                        "ErrorMessage": None,
                        "HttpStatus": response.status_code,
                        "Timestamp": request_ts.strftime('%Y-%m-%dT%H:%M:%S'),
                    },
                    "attempts": attempts,
                }
            else:
                status = "RETRY" if attempt < MAX_RETRIES else "FAILED"
                attempts.append({
                    "EventID": event_id,
                    "batch_run_id": BATCH_RUN_ID,
                    "api_type": api_type,
                    "http_method": method,
                    "api_url": url,
                    "input_value": input_value,
                    "request_body": audit_body,
                    "attempt_number": attempt,
                    "request_timestamp": request_ts,
                    "response_code": response.status_code,
                    "response_body": response.text[:4000],
                    "duration_ms": duration_ms,
                    "status": status,
                    "error_message": f"HTTP {response.status_code}",
                    "execution_context": None,
                })
                if attempt < MAX_RETRIES:
                    time.sleep(2 ** attempt)

        except requests.exceptions.RequestException as exc:
            duration_ms = int((time.time() - start_ts) * 1000)
            status = "RETRY" if attempt < MAX_RETRIES else "FAILED"
            attempts.append({
                "EventID": event_id,
                "batch_run_id": BATCH_RUN_ID,
                "api_type": api_type,
                "http_method": method,
                "api_url": url,
                "input_value": input_value,
                "request_body": json.dumps(body or data) if (body or data) else None,
                "attempt_number": attempt,
                "request_timestamp": request_ts,
                "response_code": None,
                "response_body": None,
                "duration_ms": duration_ms,
                "status": status,
                "error_message": str(exc),
                "execution_context": None,
            })
            if attempt < MAX_RETRIES:
                time.sleep(2 ** attempt)

    # Max retries exhausted
    if last_status_code and 400 <= last_status_code <= 428:
        raise RuntimeError(
            f"Non-retryable client error (HTTP {last_status_code}) after {MAX_RETRIES} attempts for {url}"
        )

    return {
        "success": False,
        "response_data": None,
        "api_metadata": {
            "Status": "FAILED",
            "ErrorMessage": f"Max retries ({MAX_RETRIES}) exhausted",
            "HttpStatus": last_status_code,
            "Timestamp": datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        },
        "attempts": attempts,
    }


def call_paginated_api(base_url, headers, api_type="employee"):
    """OData paginated caller. Follows $skiptoken via __next links.
    Accumulates all d.results across pages into a single combined result.
    Returns result dict with all attempts from all pages attached.
    """
    all_records = []
    all_attempts = []
    page_url = base_url
    page_num = 0

    while page_url:
        page_num += 1
        log.info(f"  Fetching page {page_num}...")

        page_result = call_api(
            url=page_url,
            method="GET",
            headers=headers,
            api_type=api_type,
            input_value=f"page_{page_num}",
        )
        all_attempts.extend(page_result["attempts"])

        if not page_result["success"]:
            log.error(f"  Page {page_num} FAILED. Stopping pagination.")
            return {
                "success": False,
                "response_data": all_records if all_records else None,
                "api_metadata": {
                    "Status": "FAILED",
                    "ErrorMessage": f"Pagination failed at page {page_num}",
                    "HttpStatus": page_result["api_metadata"]["HttpStatus"],
                    "Timestamp": datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
                },
                "attempts": all_attempts,
                "pages_fetched": page_num,
            }

        # Store full page response as-is (normalization handles d.results extraction)
        response_data = page_result["response_data"]
        all_records.append(response_data)
        page_count = len(response_data.get("d", {}).get("results", []))
        log.info(f"  Page {page_num}: {page_count} records in response")

        # Check for next page via __next link
        page_url = response_data.get("d", {}).get("__next", None)

    total_records = sum(len(p.get("d", {}).get("results", [])) for p in all_records)
    log.info(f"  Pagination complete. {page_num} pages, {total_records} total records.")
    return {
        "success": True,
        "response_data": all_records,
        "api_metadata": {
            "Status": "SUCCESS",
            "ErrorMessage": None,
            "HttpStatus": 200,
            "Timestamp": datetime.now().strftime('%Y-%m-%dT%H:%M:%S'),
        },
        "attempts": all_attempts,
        "pages_fetched": page_num,
        "total_records": total_records,
    }


def log_audit(result):
    """Write all attempt entries from a result to the audit logs table."""
    exec_context = json.dumps({
        "notebook_path": dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get(),
        "env": env,
    })

    for entry in result["attempts"]:
        entry["execution_context"] = exec_context

    if not result["attempts"]:
        log.warning("No audit entries to write.")
        return

    df = spark.createDataFrame(result["attempts"], schema=audit_schema)
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(LOGS_TABLE)
    log.info(f"Wrote {len(result['attempts'])} audit entries to {LOGS_TABLE}")


def write_to_bronze(result):
    """Append one row per API page to bronze. Only writes on SUCCESS with non-empty response.
    Each bronze row contains a single page response as Raw_data JSON string.
    FAILED or empty responses are captured in audit logs only — no bronze write.
    """
    if not (result["success"] and result["response_data"]):
        log.warning("Skipping bronze write: API call failed or returned empty response.")
        return

    pages = result["response_data"]
    ingestion_ts = datetime.now()
    rows = [
        Row(
            Raw_data=json.dumps(page),
            Api_metadata=json.dumps(result["api_metadata"]),
            _is_normalized=False,
            _ingestion_ts=ingestion_ts,
        )
        for page in pages
    ]

    record_count = sum(len(p.get("d", {}).get("results", [])) for p in pages if isinstance(p, dict))
    df = spark.createDataFrame(rows, schema=bronze_schema)
    df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(TARGET_BRONZE)
    log.info(f"Appended {len(rows)} rows ({record_count} records) to {TARGET_BRONZE}")

In [0]:
# ---------------------------------------------------------------------------
# Normalization: Bronze → Silver (Spark DataFrame API)
# ---------------------------------------------------------------------------


def normalize_to_silver(bronze_table, silver_table):
    """Flatten nested OData JSON from bronze using Spark DataFrame API,
    apply date transformations, and write to silver. Marks bronze rows as processed.
    """
    # --- Read un-normalized bronze records ---
    bronze_df = spark.table(bronze_table).filter(~F.col("_is_normalized"))

    if bronze_df.isEmpty():
        log.warning("No un-normalized data found in bronze. Skipping.")
        return

    # --- Parse Raw_data (one page response per row) → parse JSON → explode d.results ---
    pages_df = bronze_df.select(
        F.from_json(F.col("Raw_data"), _odata_page_schema).alias("page")
    )
    records_df = pages_df.select(
        F.explode("page.d.results").alias("rec")
    )

    # --- Explode employmentNav.results ---
    emp_df = records_df.select(
        F.col("rec.personIdExternal").alias("personIdExternal"),
        F.col("rec.firstName").alias("firstName"),
        F.col("rec.lastName").alias("lastName"),
        F.col("rec.createdDateTime").alias("createdDateTime"),
        F.col("rec.lastModifiedDateTime").alias("lastModifiedDateTime"),
        F.col("rec.personNav.userAccountNav.timeZone").alias("timeZone"),
        F.col("rec.personNav.personEmpTerminationInfoNav.latestTerminationDate").alias("terminationDate"),
        F.col("rec.personNav.emailNav.results").alias("emails"),
        F.explode("rec.personNav.employmentNav.results").alias("emp"),
    )

    # --- Explode jobInfoNav.results ---
    job_df = emp_df.select(
        "personIdExternal", "firstName", "lastName",
        "createdDateTime", "lastModifiedDateTime", "timeZone", "terminationDate", "emails",
        F.col("emp.userId").alias("UserId"),
        F.col("emp.startDate").alias("startDate"),
        F.col("emp.endDate").alias("endDate"),
        F.col("emp.isContingentWorker").alias("isContingentWorker"),
        F.col("emp.lastModifiedDateTime").alias("employmentNavLastModifiedDateTime"),
        F.col("emp.employmentId").alias("employeeId"),
        F.explode("emp.jobInfoNav.results").alias("job"),
    )

    # --- Flatten job-level fields and extract business email ---
    flat_df = job_df.select(
        F.col("personIdExternal").alias("PersonIdExternal"),
        F.col("UserId"),
        F.col("firstName"),
        F.col("lastName"),
        # Business email: emailType '2624' = business in SAP SF OData (no fallback — NULL if absent)
        F.expr("get(filter(emails, x -> x.emailType = '2624'), 0).emailAddress").alias("busEmail"),
        F.col("job.emplStatusNav.externalCode").alias("empStatus"),
        F.col("isContingentWorker"),
        F.col("createdDateTime"),
        F.col("lastModifiedDateTime"),
        F.col("startDate"),
        F.col("timeZone"),
        F.col("job.businessUnit").alias("businessUnit"),
        F.col("job.lastModifiedDateTime").alias("jobInfoNavLastModifiedDateTime"),
        F.col("endDate"),
        F.col("job.managerUserNav.email").alias("managerEmail"),
        F.col("job.jobTitle").alias("jobTitle"),
        F.col("job.positionNav.externalName_en_US").alias("positionTitle"),
        F.col("job.location").alias("location"),
        F.col("job.department").alias("department"),
        F.col("job.locationNav.lastModifiedDateTime").alias("locationNavLastModifiedDateTime"),
        F.col("job.locationNav.name").alias("locationNavName"),
        F.col("job.jobCodeNav.lastModifiedDateTime").alias("jobCodeNavLastModifiedDateTime"),
        F.col("job.jobCodeNav.name").alias("jobCodeName"),
        F.col("job.departmentNav.lastModifiedDateTime").alias("departmentNavLastModifiedDateTime"),
        F.col("job.departmentNav.name").alias("departmentNavName"),
        F.col("job.companyNav.lastModifiedDateTime").alias("companyNavLastModifiedDateTime"),
        F.col("job.companyNav.name").alias("companyNavName"),
        F.col("employmentNavLastModifiedDateTime"),
        F.col("terminationDate"),
        F.col("job.costCenter").alias("costCenter"),
        F.col("job.division").alias("division"),
        F.col("job.employeeClass").alias("employeeClass"),
        F.col("employeeId"),
    )

    # --- Apply date transformations (all at once via withColumns) ---
    # OData date format: /Date(ticks)/ → extract digits, /10000 = millis, /1000 = seconds
    DEFAULT_TS = F.lit("1900-01-01 00:00:00").cast("timestamp")
    # Dynamically compute Talend JVM server timezone offset (DST-aware).
    # Mirrors Java: TimeZone.getDefault().getOffset(System.currentTimeMillis())
    # Applied to all datetime fields except endDate to match legacy output.
    _talend_tz = ZoneInfo("America/Los_Angeles")
    _TALEND_TZ_OFFSET_MS = int(datetime.now(_talend_tz).utcoffset().total_seconds() * 1000)


    def _parse_tick_date(col_name):
        """
        Replicates Talend logic for fields using:

            new Date(
                (Long.parseLong(
                    input_row.<field>.replaceAll("[^\\d]", "")
                ) / 10000)
                - TimeZone.getDefault().getOffset(System.currentTimeMillis())
            )

        Talend JVM timezone: UTC-7.
        """

        all_digits = (
            F.regexp_replace(
                F.col(col_name),
                r"[^\d]",
                ""
            ).cast("long")
        )

        # Java Long division by 10000
        millis = F.floor(all_digits / F.lit(10000))

        # Talend:
        # millis - TimeZone.getDefault().getOffset(...)
        adjusted_millis = millis# - F.lit(_TALEND_TZ_OFFSET_MS)

        return F.when(
            F.col(col_name).isNull() | (F.col(col_name) == ""),
            DEFAULT_TS
        ).otherwise(
            F.from_unixtime(
                adjusted_millis / F.lit(1000)
            ).cast("timestamp")
    )

    def _parse_millis_ts(col_name, adjust_tz=False):
        """
        Replicates Talend logic for date-only fields:
            new Date(Long.parseLong(input_row.<field>.replaceAll("[^\\d]", ""))
                     - TimeZone.getDefault().getOffset(...)  // only for startDate, terminationDate
            )
        endDate has NO tz subtraction. Default 1900-01-01 if null/empty.
        """
        millis = F.regexp_replace(F.col(col_name), r"[^\d]", "").cast("long")
        millis = (millis - F.lit(_TALEND_TZ_OFFSET_MS)) if adjust_tz else millis
        ts_expr = (millis / 1000).cast("timestamp")
        return F.when(F.col(col_name).isNull() | (F.col(col_name) == ""), DEFAULT_TS).otherwise(ts_expr)

    def _format_iso_date(col_name, adjust_tz=False):
        """Parse /Date(millis)/ → yyyy-MM-ddT00:00:00+00:00. Default 1900-01-01T00:00:00+00:00."""
        ts = _parse_millis_ts(col_name, adjust_tz=adjust_tz)
        return F.concat(F.date_format(ts, "yyyy-MM-dd"), F.lit("T00:00:00+00:00"))

    def _format_iso_date_or_never(col_name, adjust_tz=False):
        """Parse /Date(millis)/ → 'Never' if date > 9900-01-01, else yyyy-MM-ddT00:00:00+00:00."""
        ts = _parse_millis_ts(col_name, adjust_tz=adjust_tz)
        formatted = F.concat(F.date_format(ts, "yyyy-MM-dd"), F.lit("T00:00:00+00:00"))
        return F.when(ts > F.lit("9900-01-01").cast("timestamp"), F.lit("Never")).otherwise(formatted)

    # Employee class code → label mapping (from Talend employeeClassMapper)
    _emp_class_map = F.create_map(
        F.lit("1260"), F.lit("P"),
        F.lit("1261"), F.lit("T"),
        F.lit("1262"), F.lit("N"),
        F.lit("1264"), F.lit("I"),
        F.lit("1265"), F.lit("B"),
    )

    # Apply all transformations in a single withColumns call
    df = flat_df.withColumns({
        # Tick-based datetime columns → TimestampType
        "createdDateTime": _parse_tick_date("createdDateTime"),
        "lastModifiedDateTime": _parse_tick_date("lastModifiedDateTime"),
        "jobInfoNavLastModifiedDateTime": _parse_tick_date("jobInfoNavLastModifiedDateTime"),
        "locationNavLastModifiedDateTime": _parse_tick_date("locationNavLastModifiedDateTime"),
        "jobCodeNavLastModifiedDateTime": _parse_tick_date("jobCodeNavLastModifiedDateTime"),
        "departmentNavLastModifiedDateTime": _parse_tick_date("departmentNavLastModifiedDateTime"),
        "companyNavLastModifiedDateTime": _parse_tick_date("companyNavLastModifiedDateTime"),
        "employmentNavLastModifiedDateTime": _parse_tick_date("employmentNavLastModifiedDateTime"),
        # Direct-millis date columns → ISO 8601 string (yyyy-MM-ddT00:00:00+00:00)
        "startDate": _format_iso_date("startDate", adjust_tz=False),
        "endDate": _format_iso_date_or_never("endDate"),  # No tz offset (matches Talend)
        "terminationDate": _format_iso_date_or_never("terminationDate", adjust_tz=False),
        # Strip trailing zero-width space (U+200B) from text fields
        "jobTitle": F.regexp_replace(F.col("jobTitle"), "\u200b$", ""),
        "positionTitle": F.regexp_replace(F.col("positionTitle"), "\u200b$", ""),
        "jobCodeName": F.regexp_replace(F.col("jobCodeName"), "\u200b$", ""),
        # Employee class mapping (1260→P, 1261→T, 1262→N, 1264→I, 1265→B, null→Unknown)
        "employeeClass": F.when(F.col("employeeClass").isNull(), F.lit(None))
                          .otherwise(F.coalesce(_emp_class_map[F.col("employeeClass")], F.lit(""))),
        # LoadDateUTC
        "LoadDateUTC": F.current_timestamp()
    })

    # Cast to silver schema and select in order
    df = df.select([F.col(f.name).cast(f.dataType).alias(f.name) for f in silver_schema.fields])

    # --- Write to silver ---
    record_count = df.count()
    df.write.format("delta").mode("overwrite").saveAsTable(silver_table)
    log.info(f"  Wrote {record_count} records to {silver_table}")

    # --- Mark bronze rows as normalized ---
    DeltaTable.forName(spark, bronze_table).update(
        condition="_is_normalized = false",
        set={"_is_normalized": "true"}
    )
    log.info(f"  Marked bronze rows as normalized in {bronze_table}")

In [0]:
# ---------------------------------------------------------------------------
# Main Execution
# ---------------------------------------------------------------------------
if __name__ == "__main__":

    # --- Step 1: Get SAML Assertion from IDP ---
    log.info("Step 1: Requesting SAML assertion from IDP...")
    idp_result = call_api(SESSION_APIM, method="POST", headers=HEADERS, api_type="idp")
    log_audit(idp_result)

    if not idp_result["success"]:
        raise RuntimeError(f"IDP assertion FAILED: {idp_result['api_metadata']['ErrorMessage']}")

    # IDP returns raw SAML assertion (not JSON-wrapped)
    assertion = idp_result["response_data"]
    log.info("  SAML assertion obtained.")

    # --- Step 2: Exchange Assertion for OAuth Token ---
    log.info("Step 2: Exchanging assertion for OAuth token...")
    token_body = {"assertion": assertion}
    token_result = call_api(AUTH_APIM, method="POST", headers=HEADERS, params=token_body, api_type="token")
    log_audit(token_result)

    if not token_result["success"]:
        raise RuntimeError(f"Token exchange FAILED: {token_result['api_metadata']['ErrorMessage']}")
    access_token = token_result["response_data"]["access_token"]
    log.info("  OAuth token acquired.")

    # --- Step 3: Fetch Employee Data (Paginated) ---
    log.info("Step 3: Fetching employee data (paginated)...")
    HEADERS["Authorization"]=f"Bearer {access_token}"
    employee_result = call_paginated_api(EMPLOYEE_APIM, headers=HEADERS, api_type="employee")
    log_audit(employee_result)
    write_to_bronze(employee_result)

    log.info(f"  API fetch complete. Pages: {employee_result.get('pages_fetched', 0)}, Records: {employee_result.get('total_records', 0)}.")

    # --- Step 4: Normalize Bronze → Silver ---
    log.info("Step 4: Normalizing bronze to silver...")
    normalize_to_silver(TARGET_BRONZE, TARGET_SILVER)

    log.info(f"  Pipeline completed successfully. Batch Run ID: {BATCH_RUN_ID}")